# Animation & Live Updates

Solutions can be updated interactively using `scene.redraw()` or animated over stored timesteps using the `Animation` wrapper.

## Live Redrawing with Parameters

Update a scene when an NGSolve `Parameter` changes:

In [ ]:
from ngsolve import *
from ngsolve_webgpu.jupyter import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
p = Parameter(10)
scene = Draw(sin(p * x) * sin(10 * y), mesh, order=3)

In [ ]:
# Change parameter and redraw
p.Set(20)
scene.redraw()

`scene.redraw()` re-evaluates all CoefficientFunctions and updates the GPU buffers. This works for any parameter or GridFunction change.

## Interactive Widgets

Combine with ipywidgets for sliders:

In [ ]:
from ngsolve import *
from ngsolve_webgpu.jupyter import Draw
import ipywidgets as widgets
from IPython.display import display

mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
p = Parameter(10)
scene = Draw(sin(p * x) * sin(10 * y), mesh, order=3)

slider = widgets.IntSlider(value=10, min=1, max=30, description="Frequency")
display(slider)

def on_change(change):
    p.Set(change["new"])
    scene.redraw()

slider.observe(on_change, names="value")

## Timestep Animation

Record and replay timesteps using `Animation`:

In [ ]:
from ngsolve import *
from ngsolve_webgpu.mesh import MeshData
from ngsolve_webgpu.cf import FunctionData, CFRenderer
from ngsolve_webgpu.animate import Animation
from webgpu.colormap import Colorbar
from webgpu.jupyter import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
gf = GridFunction(H1(mesh, order=3))
t = Parameter(0)
f = sin(10 * (x - 0.1 * t))

gf.Interpolate(f)

md = MeshData(mesh)
fd = FunctionData(md, gf, order=3)
cfr = CFRenderer(fd)
cfr.colormap.set_min_max(-1, 1)

ani = Animation(cfr)
scene = Draw([ani, Colorbar(cfr.colormap)])
ani.add_options_to_gui(scene.gui)
ani.add_time()

In [ ]:
# Record 10 timesteps
for tval in range(1, 101):
    t.Set(tval)
    gf.Interpolate(f)
    store = tval % 10 == 0
    if store:
        ani.add_time()
    scene.redraw(store)

`Animation` wraps a renderer and crawls its CoefficientFunction tree for `GridFunction` and `Parameter` objects. Call `ani.add_time()` to snapshot the current state. A slider appears in the GUI to scrub through stored frames.